CNN model

In [ ]:
def build_baseline_cnn(num_classes, dropout=0.4):
    model = models.Sequential([
        layers.Input(shape=IMG_SHAPE),

        layers.Conv2D(32, 3, padding="same"), layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(32, 3, padding="same"), layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, padding="same"), layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(64, 3, padding="same"), layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, padding="same"), layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(128, 3, padding="same"), layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling2D(2),

        layers.Conv2D(256, 3, padding="same"), layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(256, 3, padding="same"), layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPooling2D(2),

        layers.GlobalAveragePooling2D(),
        layers.Dropout(dropout),
        layers.Dense(128, activation="relu"),
        layers.Dropout(dropout),
        layers.Dense(num_classes, activation="softmax"),
    ], name="BaselineCNN")
    return model

In [ ]:
baseline_cnn.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

baseline_ckpt_path = str(MODELS_DIR / "baseline_best.keras")
baseline_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=6, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", mode="min", factor=0.5, patience=2, min_lr=1e-6),
    callbacks.ModelCheckpoint(baseline_ckpt_path, monitor="val_accuracy", mode="max", save_best_only=True),
]

baseline_history = baseline_cnn.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    class_weight=CLASS_WEIGHT_DICT,
    callbacks=baseline_callbacks,
)

pd.DataFrame(baseline_history.history).to_csv(MODELS_DIR / "baseline_log.csv", index=False)
print(f"\nBest baseline checkpoint saved to: {baseline_ckpt_path}")
